In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import numpy as np

In [3]:
import sqlalchemy as sa
from sqlalchemy.orm import Session
from crmprtd import setup_logging
from pycds import Station, History

save_path = './comparison_forms/'

db_url = "postgresql://tongli1997@db.pcic.uvic.ca:5433/crmp?keepalives=1&keepalives_idle=300&keepalives_interval=300&keepalives_count=9&passfile=/workspaces/crmprtd/.pgpass"
log_file_path = save_path

engine = sa.create_engine(db_url, echo=False)
session = Session(engine)

session

### The Network Name in db and Ted's spread

In [4]:
path = '/workspaces/crmprtd/update_round2/input_tables/'
df = pd.read_excel(path + '20251223-Metadata-AllRequiredChanges_round2.xlsx', header = 1)   # pandas automatically uses openpyxl

df["Station ID"] = pd.to_numeric(df["Station ID"], errors="coerce").astype("Int64")


pattern = r"Additional:\s*Replaces.*Concatenate\s+data"

df_filtered = df[df['ISSUE'].str.contains(pattern, case=False, na=False)]

df_pcds = df_filtered[['ISSUE', 'Network ID' ,'Unique Names','Unique Location (String)', 'Native ID']].reset_index(drop=True)

df_pcds['Station_name'] = df_filtered['Unique Names'].str.split('->', n=1, expand=True)[1].reset_index(drop=True).str.strip()
df_pcds = df_pcds.drop(columns=['Unique Names'])

df_pcds['native_id'] = df_filtered['Native ID'].str.split('->', n=1, expand=True)[1].reset_index(drop=True).str.strip()
df_pcds = df_pcds.drop(columns=['Native ID'])

# df_pcds["native_id"] = pd.to_numeric(df_pcds["native_id"], errors="coerce").astype("Int64")

import re

def parse_lat_lon_elev(text):
    # regex to capture: lon value, W/E, lat value, N/S, elevation
    pattern = r'([\d\.]+)\s*([WE]),\s*([\d\.]+)\s*([NS]),\s*Elev\.\s*([\d\.]+)'
    m = re.search(pattern, text)
    
    if not m:
        return None, None, None

    lon_val, lon_dir, lat_val, lat_dir, elev_val = m.groups()

    # Convert numbers
    lon = float(lon_val) * (-1 if lon_dir == "W" else 1)
    lat = float(lat_val) * (-1 if lat_dir == "S" else 1)
    elev = float(elev_val)

    return lat, lon, elev

# Apply to dataframe
df_pcds[['lat', 'lon', 'Elevation']] = df_pcds['Unique Location (String)'].apply(
    lambda x: pd.Series(parse_lat_lon_elev(x))
)
df_pcds = df_pcds.drop(columns=['Unique Location (String)'])

df_pcds['old_native_id'] = df_pcds['ISSUE'].str.extract(
    r"Replaces\s+([^-]+)\s*-",
    expand=False
).str.strip()

df_pcds

df_pcds

,ISSUE,Network ID,Station_name,native_id,lat,lon,Elevation,old_native_id
0,Additional: Replaces SBC19 - Concatenate data...,10.0,Vernon BX,AGOKNGVNBX,50.290100,-119.231000,515.0,SBC19
1,Additional: Replaces 24547 - Concatenate data...,10.0,Oliver (Panorama),AGOKNGPNRA,49.176400,-119.561900,328.0,24547
2,Additional: Replaces 25672 - Concatenate Data ...,10.0,Oliver South,AGOKNGOLVS,49.108600,-119.571500,289.0,25672
3,Additional: Replaces bc65 - Concatenate data r...,10.0,Sumas Prairie,AGBCSUMASP,49.051800,-122.140600,9.0,bc65
4,Additional: Replaces bo107 - Concatenate Data ...,10.0,Bonaparte,AGBP002,50.744722,-121.279444,446.0,bo107
5,Additional: Replaces ch107 - Concatenate Data ...,10.0,Chase,AGCH005,50.803889,-119.708333,355.0,ch107
6,Additional: Replaces co107 - Concatenate Data ...,10.0,Coldwater,AGCW006,50.011667,-120.891111,718.0,co107
7,Additional: Replaces de107 - Concatenate Data ...,10.0,Deep Creek,AGDC007,50.627500,-119.194167,517.0,de107
8,Additional: Replaces di107 - Concatenate Data ...,10.0,Diamond S (Lilloett),AGDS008,50.851667,-121.867222,450.0,di107
9,Additional: Replaces Do107 - Concatenate Data ...,10.0,Douglas Lake,AGDL009,50.204722,-120.024722,952.0,Do107


My flow: 
1. Check according to the new native_id and station_name, if not exits, insert these stations;

2. Check if the old id exist, if yes, copy the obs records to the new Station;

3. Delete the old ones? (Cause it marked as replace).

In [5]:
from sqlalchemy import text

exists_sql = text("""
SELECT EXISTS (
    SELECT 1
    FROM meta_history m
    JOIN meta_station s ON m.station_id = s.station_id
    WHERE s.native_id = :native_id
      AND m.station_name = :station_name
)
""")

for i, row in df_pcds.iterrows():
    native_id = row['native_id']
    station_name = row['Station_name']

    with engine.connect() as conn:
        exists = conn.execute(
            exists_sql,
            {
                "native_id": native_id,
                "station_name": station_name
            }
        ).scalar()

    status = "EXISTS" if exists else "NOT EXISTS"

    print(
        f"[{i+1:>3}/{len(df_pcds)}] "
        f"native_id={native_id:<15} | "
        f"station_name='{station_name}' → {status}"
    )

[  1/22] native_id=AGOKNGVNBX      | station_name='Vernon BX' → NOT EXISTS
[  2/22] native_id=AGOKNGPNRA      | station_name='Oliver (Panorama)' → NOT EXISTS
[  3/22] native_id=AGOKNGOLVS      | station_name='Oliver South' → NOT EXISTS
[  4/22] native_id=AGBCSUMASP      | station_name='Sumas Prairie' → NOT EXISTS
[  5/22] native_id=AGBP002         | station_name='Bonaparte' → NOT EXISTS
[  6/22] native_id=AGCH005         | station_name='Chase' → NOT EXISTS
[  7/22] native_id=AGCW006         | station_name='Coldwater' → NOT EXISTS
[  8/22] native_id=AGDC007         | station_name='Deep Creek' → NOT EXISTS
[  9/22] native_id=AGDS008         | station_name='Diamond S (Lilloett)' → NOT EXISTS
[ 10/22] native_id=AGDL009         | station_name='Douglas Lake' → NOT EXISTS
[ 11/22] native_id=AGGR012         | station_name='Grindrod' → NOT EXISTS
[ 12/22] native_id=AGHR013         | station_name='Halfway Ranch (Lytton-Lilloett)' → NOT EXISTS
[ 13/22] native_id=AGHC014         | station_name='Ha

### Insert all these stations

In [6]:
df_pcds

,ISSUE,Network ID,Station_name,native_id,lat,lon,Elevation,old_native_id
0,Additional: Replaces SBC19 - Concatenate data...,10.0,Vernon BX,AGOKNGVNBX,50.290100,-119.231000,515.0,SBC19
1,Additional: Replaces 24547 - Concatenate data...,10.0,Oliver (Panorama),AGOKNGPNRA,49.176400,-119.561900,328.0,24547
2,Additional: Replaces 25672 - Concatenate Data ...,10.0,Oliver South,AGOKNGOLVS,49.108600,-119.571500,289.0,25672
3,Additional: Replaces bc65 - Concatenate data r...,10.0,Sumas Prairie,AGBCSUMASP,49.051800,-122.140600,9.0,bc65
4,Additional: Replaces bo107 - Concatenate Data ...,10.0,Bonaparte,AGBP002,50.744722,-121.279444,446.0,bo107
5,Additional: Replaces ch107 - Concatenate Data ...,10.0,Chase,AGCH005,50.803889,-119.708333,355.0,ch107
6,Additional: Replaces co107 - Concatenate Data ...,10.0,Coldwater,AGCW006,50.011667,-120.891111,718.0,co107
7,Additional: Replaces de107 - Concatenate Data ...,10.0,Deep Creek,AGDC007,50.627500,-119.194167,517.0,de107
8,Additional: Replaces di107 - Concatenate Data ...,10.0,Diamond S (Lilloett),AGDS008,50.851667,-121.867222,450.0,di107
9,Additional: Replaces Do107 - Concatenate Data ...,10.0,Douglas Lake,AGDL009,50.204722,-120.024722,952.0,Do107


In [7]:
from sqlalchemy import func

stations_created = []

# for _, row in df_pcds.iloc[0:2].iterrows():
for _, row in df_pcds.iterrows():
    
    name = row['Station_name']
    nid  = row['native_id']

    # 1. Create Station
    st = Station(
        native_id=nid,
        publish=True,
        network_id=row['Network ID'])


    session.add(st)
    session.flush()  # 得到 st.id

    h = History(
        station_id=st.id,
        station_name=name,
        lat=row['lat'],
        lon=row['lon'],
        elevation=row['Elevation'],
        province="BC",
        country="CA",
        the_geom=func.ST_SetSRID(func.ST_MakePoint(row['lon'], row['lat']), 4326))

    session.add(h)

    stations_created.append((name, st.id))

session.commit()

print("Created:", stations_created)

Created: [('Vernon BX', 14936), ('Oliver (Panorama)', 14937), ('Oliver South', 14938), ('Sumas Prairie', 14939), ('Bonaparte', 14940), ('Chase', 14941), ('Coldwater', 14942), ('Deep Creek', 14943), ('Diamond S (Lilloett)', 14944), ('Douglas Lake', 14945), ('Grindrod', 14946), ('Halfway Ranch (Lytton-Lilloett)', 14947), ('Hat Creek (DFO)', 14948), ('Mamit Lake', 14949), ('Mable Lake', 14950), ('Quilchena', 14951), ('Oyama', 14952), ('Keremeos', 14953), ('Summerland Upper South', 14954), ('Silver Hills Ranch (Cherryville)', 14955), ('South Thompson', 14956), ('Sunshine Valley', 14957)]


The insert time is 2026-01-30 11:53

### Check if the old station exists

In [8]:
from sqlalchemy import text

exists_sql = text("""
SELECT EXISTS (
    SELECT 1
    FROM meta_history m
    JOIN meta_station s ON m.station_id = s.station_id
    WHERE s.native_id = :native_id
    AND s.network_id = :network_id
)
""")

exists_list = []

with engine.connect() as conn:   # reuse ONE connection
    for i, row in df_pcds.iterrows():
        native_id = row['old_native_id']
        network_id = row['Network ID']

        exists = conn.execute(
            exists_sql,
            {"native_id": native_id, "network_id": network_id}
        ).scalar()

        status = "EXISTS" if exists else "NOT EXISTS"
        exists_list.append(status)

        print(
            f"[{i+1:>3}/{len(df_pcds)}] "
            f"old_native_id={native_id:<15} | "
            f"→ {status}"
        )

# add column to DataFrame
df_pcds['old_station_status'] = exists_list

[  1/22] old_native_id=SBC19           | → EXISTS
[  2/22] old_native_id=24547           | → EXISTS
[  3/22] old_native_id=25672           | → NOT EXISTS
[  4/22] old_native_id=bc65            | → EXISTS
[  5/22] old_native_id=bo107           | → NOT EXISTS
[  6/22] old_native_id=ch107           | → NOT EXISTS
[  7/22] old_native_id=co107           | → NOT EXISTS
[  8/22] old_native_id=de107           | → NOT EXISTS
[  9/22] old_native_id=di107           | → NOT EXISTS
[ 10/22] old_native_id=Do107           | → NOT EXISTS
[ 11/22] old_native_id=gr107           | → NOT EXISTS
[ 12/22] old_native_id=ha107           | → NOT EXISTS
[ 13/22] old_native_id=hat107          | → NOT EXISTS
[ 14/22] old_native_id=ma107           | → NOT EXISTS
[ 15/22] old_native_id=mab107          | → NOT EXISTS
[ 16/22] old_native_id=qu107           | → NOT EXISTS
[ 17/22] old_native_id=SBC15           | → NOT EXISTS
[ 18/22] old_native_id=SBC25           | → NOT EXISTS
[ 19/22] old_native_id=SBC35           |

In [9]:
df_old_exists = df_pcds[df_pcds['old_station_status'] == "EXISTS"]

df_old_exists

#### Move the data from old stations to new stations

,ISSUE,Network ID,Station_name,native_id,lat,lon,Elevation,old_native_id,old_station_status
0,Additional: Replaces SBC19 - Concatenate data...,10.0,Vernon BX,AGOKNGVNBX,50.2901,-119.2310,515.0,SBC19,EXISTS
1,Additional: Replaces 24547 - Concatenate data...,10.0,Oliver (Panorama),AGOKNGPNRA,49.1764,-119.5619,328.0,24547,EXISTS
3,Additional: Replaces bc65 - Concatenate data r...,10.0,Sumas Prairie,AGBCSUMASP,49.0518,-122.1406,9.0,bc65,EXISTS
18,Additional: Replaces SBC35 - Concatenate data,10.0,Summerland Upper South,AGOKNGSLUS,49.5675,-119.6796,566.0,SBC35,EXISTS


SBC19, 24547, bc65, SBC35	

In [10]:
from sqlalchemy import text

SQL_GET_HISTORY = text("""
SELECT h.history_id
FROM meta_history h
JOIN meta_station s ON h.station_id = s.station_id
WHERE s.native_id = :native_id
  AND s.network_id = :network_id
ORDER BY h.history_id
LIMIT 1
""")

SQL_MOVE_OBS = text("""
UPDATE obs_raw
SET history_id = :new_hist_id
WHERE history_id = :old_hist_id
""")

def move_station_data(old_native_id, new_native_id, old_network_id, new_network_id, engine):
    with engine.begin() as conn:
        # get old history_id
        old_hist_id = conn.execute(
            SQL_GET_HISTORY,
            {"native_id": old_native_id, "network_id": old_network_id}
        ).scalar()

        # get new history_id
        new_hist_id = conn.execute(
            SQL_GET_HISTORY,
            {"native_id": new_native_id, "network_id": new_network_id}
        ).scalar()

        if old_hist_id is None:
            print(f"❌ Old station {old_native_id} (network {old_network_id}) has no history")
            return 0

        if new_hist_id is None:
            print(f"❌ New station {new_native_id} (network {new_network_id}) has no history")
            return 0

        # move observations
        result = conn.execute(
            SQL_MOVE_OBS,
            {
                "old_hist_id": old_hist_id,
                "new_hist_id": new_hist_id,
            }
        )

        return result.rowcount



results = []

for i, row in df_old_exists.iterrows():
    old_id  = row["old_native_id"]
    new_id  = row["native_id"]
    old_net = row["Network ID"]
    new_net = row["Network ID"]

    print(f"[{i+1}/{len(df_old_exists)}] Moving {old_id} (net {old_net}) → {new_id} (net {new_net})")

    n_moved = move_station_data(old_id, new_id, old_net, new_net, engine)

    print(f"    ✔ Moved {n_moved} rows")
    results.append(n_moved)

df_old_exists["n_moved"] = results

[1/4] Moving SBC19 (net 10.0) → AGOKNGVNBX (net 10.0)
    ✔ Moved 17254 rows
[2/4] Moving 24547 (net 10.0) → AGOKNGPNRA (net 10.0)
    ✔ Moved 15768 rows
[4/4] Moving bc65 (net 10.0) → AGBCSUMASP (net 10.0)
    ✔ Moved 9185 rows
[19/4] Moving SBC35 (net 10.0) → AGOKNGSLUS (net 10.0)
    ✔ Moved 17488 rows


/tmp/ipykernel_15823/1156388578.py:69: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_old_exists["n_moved"] = results


### Can delete the four old stations? The other old stations seem not exist. Only need to delete meta_history and meta_station I think


In [11]:
SQL_GET_STATION_ID = text("""
SELECT station_id
FROM meta_station
WHERE native_id = :native_id
""")

SQL_DELETE_OBS_derived = sa.text("""
DELETE FROM obs_derived_values dv
USING meta_history h
WHERE dv.history_id = h.history_id
  AND h.station_id = :station_id
""")

SQL_DELETE_HISTORY = text("""
DELETE FROM meta_history
WHERE station_id = :station_id
""")

SQL_DELETE_STATION = text("""
DELETE FROM meta_station
WHERE station_id = :station_id
""")


results = []

for i, row in df_old_exists.iterrows():
    old_native_id = row['old_native_id']

    with engine.begin() as conn:
        station_id = conn.execute(
            SQL_GET_STATION_ID,
            {"native_id": old_native_id}
        ).scalar()

        if station_id is None:
            print(f"[{i+1}] old_native_id={old_native_id} → station not found, skipping")
            continue

        print(f"[{i+1}] Deleting station '{old_native_id}' (station_id={station_id})")
        n0 = conn.execute(SQL_DELETE_OBS_derived, {"station_id": station_id}).rowcount
        n1 = conn.execute(SQL_DELETE_HISTORY, {"station_id": station_id}).rowcount
        n2 = conn.execute(SQL_DELETE_STATION, {"station_id": station_id}).rowcount

        print(
            f"deleted obs_derived={n0}, "
            f"deleted history={n1}, "
            f"deleted station={n2}, "
        )

        results.append({
            "old_native_id": old_native_id,
            "station_id": station_id,
            "obs_derived_deleted": n0,
            "history_deleted": n1,
            "station_deleted": n2,

        })

[1] Deleting station 'SBC19' (station_id=3102)
deleted obs_derived=20, deleted history=1, deleted station=1, 
[2] Deleting station '24547' (station_id=1523)
deleted obs_derived=13, deleted history=1, deleted station=1, 
[4] Deleting station 'bc65' (station_id=1538)
deleted obs_derived=28, deleted history=1, deleted station=1, 
[19] Deleting station 'SBC35' (station_id=3110)
deleted obs_derived=24, deleted history=1, deleted station=1, 
